<a href="https://colab.research.google.com/github/Vigneshvar16/CSA0607-DAA/blob/main/CO4_A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# DAA CO4A - Vigneshvar S (192424187) |
import heapq

# ---------- Task 1: fractional loading (greedy by value/weight) ----------
def fractional_load(items, cap):                 # items: (id, weight, value)
    rem, total, taken = cap, 0.0, []
    for i, w, v in sorted(items, key=lambda t: t[2] / t[1], reverse=True):
        if rem == 0: break
        f = 1.0 if w <= rem else rem / w         # whole package, else the fraction that fits
        taken.append((i, round(f, 3))); total += f * v; rem -= f * w
    return taken, round(total, 2)

# ---------- Task 2: job sequencing with deadlines ----------
def job_seq(jobs):                               # jobs: (id, deadline, profit)
    D = max(d for _, d, _ in jobs); slot = [None] * (D + 1); total = 0
    for jid, d, p in sorted(jobs, key=lambda t: t[2], reverse=True):
        for s in range(min(d, D), 0, -1):        # latest free slot <= deadline
            if slot[s] is None: slot[s] = jid; total += p; break
    return slot[1:], total

# ---------- Task 3: 0/1 knapsack with reconstruction ----------
def knapsack(items, cap):                        # items: (id, weight, value)
    n = len(items); dp = [[0] * (cap + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        _, w, v = items[i - 1]
        for c in range(cap + 1):
            dp[i][c] = dp[i - 1][c]                              # exclude
            if w <= c: dp[i][c] = max(dp[i][c], v + dp[i - 1][c - w])   # include
    chosen, c = [], cap
    for i in range(n, 0, -1):
        if dp[i][c] != dp[i - 1][c]: chosen.append(items[i - 1][0]); c -= items[i - 1][1]
    return dp, dp[n][cap], chosen[::-1]

# ---------- Task 4: word break (faulty vs repaired) ----------
def word_break(text, dic, repaired):
    n = len(text); dp = [True] + [False] * n
    for i in range(1, n + 1):
        for j in range(i):
            if (dp[j] or not repaired) and text[j:i] in dic:     # repair = extra test dp[j]
                dp[i] = True; break
    return dp[n]

# ---------- Task 5: shortest paths ----------
E = [('A','B',4),('A','C',7),('B','C',2),('B','D',6),('C','D',1),('D','E',3),('C','E',8)]
V = ['A','B','C','D','E']

def dijkstra(src):
    dist = {v: float('inf') for v in V}; pred = {v: None for v in V}; dist[src] = 0
    pq = [(0, src)]; done = set()
    while pq:
        d, u = heapq.heappop(pq)                 # select unvisited vertex with least distance
        if u in done: continue
        done.add(u)
        for a, b, w in E:
            if a == u and d + w < dist[b]:       # relaxation
                dist[b] = d + w; pred[b] = u; heapq.heappush(pq, (dist[b], b))
    return dist, pred

def bellman_ford(src):
    dist = {v: float('inf') for v in V}; dist[src] = 0
    for _ in range(len(V) - 1):
        for a, b, w in E:
            if dist[a] + w < dist[b]: dist[b] = dist[a] + w
    neg_cycle = any(dist[a] + w < dist[b] for a, b, w in E)     # extra pass
    return dist, neg_cycle

def floyd():
    INF = float('inf'); D = {i: {j: (0 if i == j else INF) for j in V} for i in V}
    for a, b, w in E: D[a][b] = w
    for k in V:
        for i in V:
            for j in V:
                if D[i][k] + D[k][j] < D[i][j]: D[i][j] = D[i][k] + D[k][j]
    return D

# ---------- Task 6: weighted job (interval) scheduling, DP ----------
def schedule(jobs):                              # jobs: (name, start, finish, value)
    jobs = sorted(jobs, key=lambda j: j[2]); n = len(jobs)
    p = [0] * (n + 1)
    for j in range(1, n + 1):                    # p[j] = last job finishing <= start of job j
        for i in range(j - 1, 0, -1):
            if jobs[i - 1][2] <= jobs[j - 1][1]: p[j] = i; break
    opt = [0] * (n + 1)
    for j in range(1, n + 1):
        opt[j] = max(opt[j - 1], jobs[j - 1][3] + opt[p[j]])
    sel, j = [], n
    while j > 0:
        if jobs[j - 1][3] + opt[p[j]] > opt[j - 1]: sel.append(jobs[j - 1][0]); j = p[j]
        else: j -= 1
    return p[1:], opt, sel[::-1], opt[n]

if __name__ == "__main__":
    print("TASK 1:", fractional_load([("P1",10,60),("P2",20,100),("P3",30,120)], 50))
    J = [("A",2,80),("B",1,35),("C",2,60),("D",3,40),("E",1,25)]
    print("TASK 2:", job_seq(J))
    J2 = [("A",2,80),("B",1,70),("C",2,60),("D",3,40),("E",1,25)]
    print("TASK 2 (B profit 70):", job_seq(J2))
    dp, best, chosen = knapsack([("P",2,12),("Q",3,20),("R",4,25),("S",5,30)], 7)
    print("TASK 3: optimum =", best, "items =", chosen)
    for r in dp: print("   ", r)
    print("TASK 4: 'catdog' faulty =", word_break("catdog", {"dog"}, False),
          "| repaired =", word_break("catdog", {"dog"}, True))
    print("TASK 4: 'dogdog' faulty =", word_break("dogdog", {"dog"}, False),
          "| repaired =", word_break("dogdog", {"dog"}, True))
    d, pr = dijkstra('A'); print("TASK 5 Dijkstra:", d, "pred:", pr)
    print("TASK 5 Bellman-Ford:", bellman_ford('A'))
    F = floyd(); print("TASK 5 Floyd (row A):", F['A'])
    jobs = [("J1",1,3,20),("J2",2,5,50),("J3",4,6,30),("J4",6,8,40),("J5",5,9,60),("J6",8,10,30),("J7",9,11,50),("J8",11,13,40)]
    print("TASK 6 given data:", schedule(jobs))
    adv = [("A",1,10,100),("B",1,4,60),("C",4,7,60),("D",7,10,60),("E",2,5,30)]
    print("TASK 6 adversarial:", schedule(adv))
    print("   highest-value-first would give only 100 (job A blocks all others)")

TASK 1: ([('P1', 1.0), ('P2', 1.0), ('P3', 0.667)], 240.0)
TASK 2: (['C', 'A', 'D'], 180)
TASK 2 (B profit 70): (['B', 'A', 'D'], 190)
TASK 3: optimum = 45 items = ['Q', 'R']
    [0, 0, 0, 0, 0, 0, 0, 0]
    [0, 0, 12, 12, 12, 12, 12, 12]
    [0, 0, 12, 20, 20, 32, 32, 32]
    [0, 0, 12, 20, 25, 32, 37, 45]
    [0, 0, 12, 20, 25, 32, 37, 45]
TASK 4: 'catdog' faulty = True | repaired = False
TASK 4: 'dogdog' faulty = True | repaired = True
TASK 5 Dijkstra: {'A': 0, 'B': 4, 'C': 6, 'D': 7, 'E': 10} pred: {'A': None, 'B': 'A', 'C': 'B', 'D': 'C', 'E': 'D'}
TASK 5 Bellman-Ford: ({'A': 0, 'B': 4, 'C': 6, 'D': 7, 'E': 10}, False)
TASK 5 Floyd (row A): {'A': 0, 'B': 4, 'C': 6, 'D': 7, 'E': 10}
TASK 6 given data: ([0, 0, 1, 3, 2, 4, 5, 7], [0, 20, 50, 50, 90, 110, 120, 160, 200], ['J2', 'J5', 'J7', 'J8'], 200)
TASK 6 adversarial: ([0, 0, 1, 0, 3], [0, 60, 60, 120, 120, 180], ['B', 'C', 'D'], 180)
   highest-value-first would give only 100 (job A blocks all others)
